In [37]:
import pandas as pd

ppr = pd.read_csv(
    "../data/raw/PPR-ALL.csv",
    encoding="latin1"
)

print(ppr.shape, ppr.columns)

ppr.head()


(755360, 9) Index(['Date of Sale (dd/mm/yyyy)', 'Address', 'County', 'Eircode',
       'Price ()', 'Not Full Market Price', 'VAT Exclusive',
       'Description of Property', 'Property Size Description'],
      dtype='object')


C:\Users\tiarn\AppData\Local\Temp\ipykernel_7780\3329172716.py:3: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  ppr = pd.read_csv(


,Date of Sale (dd/mm/yyyy),Address,County,Eircode,Price (),Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description
0,01/01/2010,"5 Braemor Drive, Churchtown, Co.Dublin",Dublin,NaN,"343,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
1,03/01/2010,"134 Ashewood Walk, Summerhill Lane, Portlaoise",Laois,NaN,"185,000.00",No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...
2,04/01/2010,"1 Meadow Avenue, Dundrum, Dublin 14",Dublin,NaN,"438,500.00",No,No,Second-Hand Dwelling house /Apartment,NaN
3,04/01/2010,"1 The Haven, Mornington",Meath,NaN,"400,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
4,04/01/2010,"11 Melville Heights, Kilkenny",Kilkenny,NaN,"160,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN


In [38]:
print(ppr.shape)
print(ppr.columns)


(755360, 9)
Index(['Date of Sale (dd/mm/yyyy)', 'Address', 'County', 'Eircode',
       'Price ()', 'Not Full Market Price', 'VAT Exclusive',
       'Description of Property', 'Property Size Description'],
      dtype='object')


In [39]:
ppr = ppr.rename(columns={"Price ()": "Price_EUR"})
print("Price_EUR" in ppr.columns)


True


In [40]:
ppr["Price_EUR"].head(5).apply(repr)


0    '\x80343,000.00'
1    '\x80185,000.00'
2    '\x80438,500.00'
3    '\x80400,000.00'
4    '\x80160,000.00'
Name: Price_EUR, dtype: object

In [41]:
ppr["Price_EUR"] = (
    ppr["Price_EUR"]
    .astype(str)
    .str.replace(r"[^\d.]", "", regex=True)
)

ppr["Price_EUR"] = pd.to_numeric(ppr["Price_EUR"], errors="coerce")
ppr.head()

,Date of Sale (dd/mm/yyyy),Address,County,Eircode,Price_EUR,Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description
0,01/01/2010,"5 Braemor Drive, Churchtown, Co.Dublin",Dublin,NaN,343000.0,No,No,Second-Hand Dwelling house /Apartment,NaN
1,03/01/2010,"134 Ashewood Walk, Summerhill Lane, Portlaoise",Laois,NaN,185000.0,No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...
2,04/01/2010,"1 Meadow Avenue, Dundrum, Dublin 14",Dublin,NaN,438500.0,No,No,Second-Hand Dwelling house /Apartment,NaN
3,04/01/2010,"1 The Haven, Mornington",Meath,NaN,400000.0,No,No,Second-Hand Dwelling house /Apartment,NaN
4,04/01/2010,"11 Melville Heights, Kilkenny",Kilkenny,NaN,160000.0,No,No,Second-Hand Dwelling house /Apartment,NaN


In [42]:
print(ppr["Price_EUR"].describe())


count    7.553600e+05
mean     3.101274e+05
std      1.180266e+06
min      5.001000e+03
25%      1.400000e+05
50%      2.361590e+05
75%      3.535000e+05
max      3.876652e+08
Name: Price_EUR, dtype: float64


In [45]:
ppr[ppr["Price_EUR"] > 5_000_000][["County", "Price_EUR"]].sort_values("Price_EUR", ascending=False).head(10)


,County,Price_EUR
690492,Wicklow,3.876652e+08
586318,Dublin,2.250000e+08
705921,Dublin,2.219427e+08
431179,Dublin,1.823789e+08
473341,Dublin,1.701428e+08
749049,Dublin,1.693250e+08
705102,Dublin,1.423370e+08
409191,Dublin,1.422566e+08
316557,Dublin,1.391650e+08
575801,Dublin,1.358598e+08


In [46]:
ppr["Sale_Date"] = pd.to_datetime(
    ppr["Date of Sale (dd/mm/yyyy)"],
    dayfirst=True,
    errors="coerce"
)

ppr = ppr.dropna(subset=["Sale_Date"])

ppr["Year"] = ppr["Sale_Date"].dt.year
ppr["Quarter"] = ppr["Sale_Date"].dt.to_period("Q").dt.quarter
ppr["Period"] = ppr["Year"].astype(str) + " Q" + ppr["Quarter"].astype(str)


In [47]:
ppr = ppr[
    (ppr["Not Full Market Price"] == "No") &
    (ppr["VAT Exclusive"] == "No")
]
ppr.head(10)

,Date of Sale (dd/mm/yyyy),Address,County,Eircode,Price_EUR,Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description,Sale_Date,Year,Quarter,Period
0,01/01/2010,"5 Braemor Drive, Churchtown, Co.Dublin",Dublin,NaN,343000.0,No,No,Second-Hand Dwelling house /Apartment,NaN,2010-01-01,2010,1,2010 Q1
2,04/01/2010,"1 Meadow Avenue, Dundrum, Dublin 14",Dublin,NaN,438500.0,No,No,Second-Hand Dwelling house /Apartment,NaN,2010-01-04,2010,1,2010 Q1
3,04/01/2010,"1 The Haven, Mornington",Meath,NaN,400000.0,No,No,Second-Hand Dwelling house /Apartment,NaN,2010-01-04,2010,1,2010 Q1
4,04/01/2010,"11 Melville Heights, Kilkenny",Kilkenny,NaN,160000.0,No,No,Second-Hand Dwelling house /Apartment,NaN,2010-01-04,2010,1,2010 Q1
5,04/01/2010,"12 Sallymount Avenue, Ranelagh",Dublin,NaN,425000.0,No,No,Second-Hand Dwelling house /Apartment,NaN,2010-01-04,2010,1,2010 Q1
6,04/01/2010,"13 Oakleigh Wood, Dooradoyle, Limerick",Limerick,NaN,172500.0,No,No,Second-Hand Dwelling house /Apartment,NaN,2010-01-04,2010,1,2010 Q1
7,04/01/2010,"13 The Drive, Chapelstown Gate, Tullow Road",Carlow,NaN,177500.0,No,No,Second-Hand Dwelling house /Apartment,NaN,2010-01-04,2010,1,2010 Q1
8,04/01/2010,"15 Carriglawn, Waterpark, Carrigaline",Cork,NaN,180000.0,No,No,Second-Hand Dwelling house /Apartment,NaN,2010-01-04,2010,1,2010 Q1
9,04/01/2010,"15a Moore Bay, Kilkee",Clare,NaN,126500.0,No,No,Second-Hand Dwelling house /Apartment,NaN,2010-01-04,2010,1,2010 Q1
10,04/01/2010,"16 Aisling Geal, Fr. Russell Road",Limerick,NaN,110000.0,No,No,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...,2010-01-04,2010,1,2010 Q1


In [48]:
def remove_outliers(group):
    if len(group) < 20:
        return group  # examine small samples individually
    low = group["Price_EUR"].quantile(0.05)
    high = group["Price_EUR"].quantile(0.95)
    return group[(group["Price_EUR"] >= low) & (group["Price_EUR"] <= high)]

ppr_clean = (
    ppr
    .groupby(["County", "Year", "Quarter"], group_keys=False)
    .apply(remove_outliers)
)


C:\Users\tiarn\AppData\Local\Temp\ipykernel_7780\3174485106.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(remove_outliers)


In [49]:
print("Before:", ppr.shape)
print("After :", ppr_clean.shape)

ppr_clean["Price_EUR"].describe()


Before: (594846, 13)
After : (535892, 13)


count    5.358920e+05
mean     2.626858e+05
std      1.726672e+05
min      6.847310e+03
25%      1.430000e+05
50%      2.260000e+05
75%      3.360000e+05
max      1.900000e+06
Name: Price_EUR, dtype: float64

In [50]:
roi_quarterly_clean = (
    ppr_clean
    .groupby(["County", "Year", "Quarter", "Period"])
    .agg(
        transaction_count=("Price_EUR", "size"),
        avg_price=("Price_EUR", "mean")
    )
    .reset_index()
)


In [52]:
roi_quarterly_clean.head()



,County,Year,Quarter,Period,transaction_count,avg_price
0,Carlow,2010,1,2010 Q1,23,171836.956522
1,Carlow,2010,2,2010 Q2,25,177037.000000
2,Carlow,2010,3,2010 Q3,33,164772.727273
3,Carlow,2010,4,2010 Q4,31,195185.806452
4,Carlow,2011,1,2011 Q1,20,161475.000000


In [53]:
roi_quarterly_clean["avg_price"].describe()

count      1664.000000
mean     183790.611754
std       82419.930076
min       39079.447758
25%      120990.166343
50%      164756.051400
75%      227415.340467
max      548900.942681
Name: avg_price, dtype: float64

In [54]:
roi_quarterly_clean["avg_price"] = roi_quarterly_clean["avg_price"].round(2)
roi_quarterly_clean["transaction_count"] = roi_quarterly_clean["transaction_count"].astype(int)


In [55]:
roi_quarterly_clean = roi_quarterly_clean.sort_values(
    ["County", "Year", "Quarter"]
).reset_index(drop=True)


In [58]:
roi_quarterly_clean.to_csv(
    "../data/roi_quarterly_clean.csv",
    index=False
)


In [60]:
test = pd.read_csv("../data/roi_quarterly_clean.csv")
test.head()
test.shape


(1664, 6)